# Phase 3 — Predictive Modeling and Validation

This notebook develops and evaluates regression models for NYC taxi fare prediction.

The modelling workflow will eventually include:

- modelling-data validation;
- reproducible sampling;
- train/test splitting;
- baseline regression;
- Random Forest;
- Gradient Boosting;
- model comparison;
- residual analysis;
- model-based feature importance.

The full feature-engineered dataset contains more than 54 million records.
A reproducible modelling sample will be used first to validate the modelling pipeline before considering larger training datasets.

**Current scope: Phase 3A — Modeling Data Validation only.** This run inspects the first one million rows, retains every inspected row, and does not create a final modelling sample, apply eligibility rules, split data, or train/evaluate models. The later workflow listed above is deferred until validation results and handling rules are reviewed.

The nine candidate features exclude `key` (an identifier) and the raw `pickup_datetime` string (already transformed into temporal features). `fare_amount` is the target, not an input feature.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/processed/train_features.csv")
TARGET = "fare_amount"
FEATURES = [
    "trip_distance",
    "pickup_hour",
    "day_of_week",
    "is_weekend",
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
]
RANDOM_STATE = 42
VALIDATION_ROWS = 1_000_000

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")
print(f"Using dataset: {DATA_PATH.resolve()}")

Using dataset: /Users/xiaochuan/nyc-taxi-fare-prediction/data/processed/train_features.csv


## 1. Initial Modeling Sample

Read the **first** 1,000,000 rows and the target plus nine candidate features for schema validation, missing-value validation, range checks, and anomaly inspection. This deterministic prefix is not a random or final modelling sample, and its findings must not be generalized to all 54 million rows. No full-dataset scan is performed.

Inspect the original CSV header before pandas can rename duplicate columns. Only the header is read in this preflight check; missing required fields raise a clear error.

In [3]:
import csv

expected_columns = [TARGET] + FEATURES
with DATA_PATH.open(newline="", encoding="utf-8-sig") as source:
    source_columns = next(csv.reader(source))
source_duplicate_columns = pd.Index(source_columns)[pd.Index(source_columns).duplicated()].unique().tolist()
print("Duplicate column names in source header:", source_duplicate_columns)
missing_source_columns = sorted(set(expected_columns) - set(source_columns))
if missing_source_columns:
    raise ValueError(f"Missing required columns in source: {missing_source_columns}")
if source_duplicate_columns:
    raise ValueError(f"Ambiguous duplicate source column names: {source_duplicate_columns}")

sample_check = pd.read_csv(
    DATA_PATH,
    usecols=[TARGET] + FEATURES,
    nrows=VALIDATION_ROWS,
)
print("Validation sample shape:", sample_check.shape)
display(sample_check.head())

Duplicate column names in source header: []
Validation sample shape: (1000000, 10)


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,trip_distance,pickup_hour,day_of_week,is_weekend
0,4.5,-73.844311,40.721319,-73.841610,40.712278,1,1.030765,17,0,0
1,16.9,-74.016048,40.711303,-73.979268,40.782004,1,8.450145,16,1,0
2,5.7,-73.982738,40.761270,-73.991242,40.750562,2,1.389527,0,3,0
3,7.7,-73.987130,40.733143,-73.991567,40.758092,1,2.799274,4,5,1
4,5.3,-73.968095,40.768008,-73.956655,40.783762,1,1.999160,7,1,0


## 2. Schema Validation

Confirm that all required fields are present, column names are unique, and all ten fields are numeric before numerical checks. Report the actual number of rows read; no assumption of full-dataset representativeness is made.

In [4]:
expected_columns = [TARGET] + FEATURES
missing_columns = sorted(set(expected_columns) - set(sample_check.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
duplicated_columns = sample_check.columns[sample_check.columns.duplicated()].tolist()
print("Duplicated validation column names:", duplicated_columns)
if duplicated_columns:
    raise ValueError(f"Duplicated validation column names: {duplicated_columns}")
if len(sample_check.columns) != len(expected_columns):
    raise ValueError(f"Expected {len(expected_columns)} columns, got {len(sample_check.columns)}")
rows_inspected = len(sample_check)
if rows_inspected == 0:
    raise ValueError("The validation sample is empty")
print(f"Number of rows: {rows_inspected:,}")
print(f"Number of columns: {len(sample_check.columns)}")
display(sample_check.dtypes)
non_numeric_columns = [c for c in expected_columns if not pd.api.types.is_numeric_dtype(sample_check[c])]
if non_numeric_columns:
    raise ValueError(f"Non-numeric modelling fields: {non_numeric_columns}")
if rows_inspected != VALIDATION_ROWS:
    print(f"Requested {VALIDATION_ROWS:,} rows, but read {rows_inspected:,}.")

# Shared reporting helper: overlapping checks are not mutually exclusive.
# Every percentage uses the actual number of inspected rows as denominator.
def summarize_checks(checks):
    return pd.DataFrame([
        {"check": name, "count": int(mask.sum()),
         "percentage": float(mask.sum() / rows_inspected * 100)}
        for name, mask in checks.items()
    ])

Duplicated validation column names: []
Number of rows: 1,000,000
Number of columns: 10


fare_amount          float64
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
trip_distance        float64
pickup_hour            int64
day_of_week            int64
is_weekend             int64
dtype: object

## 3. Descriptive Statistics

In [5]:
descriptive_statistics = sample_check.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
).T
display(descriptive_statistics)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,99.9%,max
fare_amount,1000000.0,11.303406,9.522975,0.010000,3.300000,4.100000,6.000000,8.500000,12.500000,30.100000,52.000000,74.500000,500.000000
pickup_longitude,1000000.0,-73.975491,0.034507,-74.299372,-74.014280,-74.006860,-73.992280,-73.982094,-73.968362,-73.937035,-73.787842,-73.776706,-73.700438
pickup_latitude,1000000.0,40.750875,0.026926,40.503982,40.645413,40.710082,40.736580,40.753383,40.767543,40.787842,40.806678,40.842675,40.998754
dropoff_longitude,1000000.0,-73.974576,0.033997,-74.299372,-74.015213,-74.007439,-73.991575,-73.980606,-73.965394,-73.923932,-73.805452,-73.769402,-73.700155
dropoff_latitude,1000000.0,40.751257,0.030853,40.501978,40.646607,40.703469,40.735604,40.753871,40.768399,40.794088,40.829784,40.883380,40.998754
passenger_count,1000000.0,1.691141,1.306291,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,5.000000,6.000000,6.000000,9.000000
trip_distance,1000000.0,3.308444,3.554053,0.000000,0.000000,0.560032,1.254650,2.153391,3.914000,9.989398,20.263769,23.005441,49.484019
pickup_hour,1000000.0,13.508718,6.512035,0.000000,0.000000,1.000000,9.000000,14.000000,19.000000,22.000000,23.000000,23.000000,23.000000
day_of_week,1000000.0,3.039640,1.949768,0.000000,0.000000,0.000000,1.000000,3.000000,5.000000,6.000000,6.000000,6.000000,6.000000
is_weekend,1000000.0,0.282492,0.450212,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 4. Missing Values

In [6]:
missing_counts = sample_check.isna().sum()
missing_rates = sample_check.isna().mean() * 100
missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_rate_pct": missing_rates,
}).sort_values("missing_count", ascending=False)
display(missing_summary)
missing_rows = sample_check.isna().any(axis=1)
if missing_counts.sum() == 0:
    if rows_inspected == VALIDATION_ROWS == 1_000_000:
        print("No missing values detected in the 1M-row validation sample.")
    else:
        print(f"No missing values detected in the {rows_inspected:,}-row validation sample.")

,missing_count,missing_rate_pct
fare_amount,0,0.0
pickup_longitude,0,0.0
pickup_latitude,0,0.0
dropoff_longitude,0,0.0
dropoff_latitude,0,0.0
passenger_count,0,0.0
trip_distance,0,0.0
pickup_hour,0,0.0
day_of_week,0,0.0
is_weekend,0,0.0


No missing values detected in the 1M-row validation sample.


## 5. Passenger Count Validation

Week 2 full-data analysis identified rare counts such as 129 and 208. Inspect their presence in this prefix without assuming it contains all previously observed anomalies. Identify, count, and inspect only: do not remove or recode passenger counts.

In [7]:
passenger = sample_check["passenger_count"]
passenger_counts = passenger.value_counts().sort_index()
display(passenger_counts)
print(f"Min passenger_count: {passenger.min()}")
print(f"Max passenger_count: {passenger.max()}")
passenger_checks = {
    "passenger_count < 1": passenger < 1,
    "passenger_count == 0": passenger == 0,
    "passenger_count > 6": passenger > 6,
}
passenger_summary = summarize_checks(passenger_checks)
n_zero = int((passenger == 0).sum())
n_above_six = int((passenger > 6).sum())
display(passenger_summary)
display(sample_check.loc[
    passenger > 6,
    ["fare_amount", "trip_distance", "passenger_count", "pickup_hour", "day_of_week"],
].head(50))

passenger_count
1    693517
2    148911
3     43915
4     21514
5     70860
6     21282
9         1
Name: count, dtype: int64

Min passenger_count: 1
Max passenger_count: 9


,check,count,percentage
0,passenger_count < 1,0,0.0000
1,passenger_count == 0,0,0.0000
2,passenger_count > 6,1,0.0001


,fare_amount,trip_distance,passenger_count,pickup_hour,day_of_week
982004,104.0,13.045926,9,15,1


## 6. Fare Amount Validation

RMSE is sensitive to extreme target values because errors are squared.
The thresholds below are descriptive inspection checks, not chosen fare eligibility thresholds. No fares are removed or capped.

In [8]:
fare = sample_check[TARGET]
fare_quantiles = fare.quantile([0, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999, 1])
display(fare_quantiles)
fare_checks = {
    "fare <= 0": fare <= 0,
    "fare < 2.5": fare < 2.5,
    "fare > 50": fare > 50,
    "fare > 100": fare > 100,
    "fare > 200": fare > 200,
}
fare_summary = summarize_checks(fare_checks)
display(fare_summary)

0.000      0.01
0.010      3.30
0.050      4.10
0.500      8.50
0.950     30.10
0.990     52.00
0.999     74.50
1.000    500.00
Name: fare_amount, dtype: float64

,check,count,percentage
0,fare <= 0,0,0.0000
1,fare < 2.5,7,0.0007
2,fare > 50,12230,1.2230
3,fare > 100,195,0.0195
4,fare > 200,12,0.0012


## 7. Trip Distance Validation

`trip_distance` is Haversine straight-line distance in kilometres, not driven route length. Zero-distance trips are not necessarily invalid: possible explanations include genuine short trips, coordinate precision limitations, or anomalous records. Inspect zero, near-zero and longer-distance trips without removing any rows.

In [9]:
distance = sample_check["trip_distance"]
distance_quantiles = distance.quantile([0, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999, 1])
display(distance_quantiles)
distance_checks = {
    "trip_distance == 0": distance == 0,
    "trip_distance <= 0.1": distance <= 0.1,
    "trip_distance > 20": distance > 20,
    "trip_distance > 30": distance > 30,
}
distance_summary = summarize_checks(distance_checks)
display(distance_summary)

0.000     0.000000
0.010     0.000000
0.050     0.560032
0.500     2.153391
0.950     9.989398
0.990    20.263769
0.999    23.005441
1.000    49.484019
Name: trip_distance, dtype: float64

,check,count,percentage
0,trip_distance == 0,10441,1.0441
1,trip_distance <= 0.1,16284,1.6284
2,trip_distance > 20,11543,1.1543
3,trip_distance > 30,90,0.0090


## 8. Coordinate Range Validation

Use the inclusive Week 1 cleaning bounds: longitude −74.3 to −73.7 and latitude 40.5 to 41.0. Range violations count values below or above those limits. Missing values are reported separately; the combined row-level count counts a trip once even if multiple coordinates violate the bounds.

In [10]:
coordinate_bounds = {
    "pickup_longitude": (-74.3, -73.7),
    "dropoff_longitude": (-74.3, -73.7),
    "pickup_latitude": (40.5, 41.0),
    "dropoff_latitude": (40.5, 41.0),
}
coordinate_masks = {}
coordinate_records = []
for feature, (lower, upper) in coordinate_bounds.items():
    values = sample_check[feature]
    outside = (values < lower) | (values > upper)
    coordinate_masks[feature] = outside
    coordinate_records.append({
        "feature": feature, "min": values.min(), "max": values.max(),
        "outside_expected_range_count": int(outside.sum()),
        "outside_expected_range_pct": float(outside.mean() * 100),
    })
coordinate_summary = pd.DataFrame(coordinate_records)
coordinate_violations = pd.DataFrame(coordinate_masks).any(axis=1)
display(coordinate_summary)
print(f"Rows with coordinate range violations: {coordinate_violations.sum():,}")

,feature,min,max,outside_expected_range_count,outside_expected_range_pct
0,pickup_longitude,-74.299372,-73.700438,0,0.0
1,dropoff_longitude,-74.299372,-73.700155,0,0.0
2,pickup_latitude,40.503982,40.998754,0,0.0
3,dropoff_latitude,40.501978,40.998754,0,0.0


Rows with coordinate range violations: 0


## 9. Temporal Feature Validation

Temporal features retain the Kaggle-recorded clock time and the corrected existing interpretation. No timezone conversion is performed. Membership checks enforce integer hour/day codes, not merely numeric bounds. An additional consistency check verifies that the existing weekend flag agrees with the day-of-week definition.

In [11]:
temporal_allowed = {
    "pickup_hour": list(range(24)),
    "day_of_week": list(range(7)),
    "is_weekend": [0, 1],
}
temporal_checks = {}
for feature, allowed in temporal_allowed.items():
    print(f"{feature} unique values: {np.sort(sample_check[feature].unique()).tolist()}")
    temporal_checks[f"invalid {feature}"] = ~sample_check[feature].isin(allowed)
temporal_summary = summarize_checks(temporal_checks)
temporal_violations = pd.DataFrame(temporal_checks).any(axis=1)
display(temporal_summary)
valid_weekly_codes = sample_check["day_of_week"].isin(range(7)) & sample_check["is_weekend"].isin([0, 1])
weekend_mismatch = valid_weekly_codes & (
    sample_check["is_weekend"] != (sample_check["day_of_week"] >= 5).astype(int)
)
print(f"Weekend/day-of-week consistency violations among valid codes: {weekend_mismatch.sum():,}")

pickup_hour unique values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
day_of_week unique values: [0, 1, 2, 3, 4, 5, 6]
is_weekend unique values: [0, 1]


,check,count,percentage
0,invalid pickup_hour,0,0.0
1,invalid day_of_week,0,0.0
2,invalid is_weekend,0,0.0


Weekend/day-of-week consistency violations among valid codes: 0


## 10. Finite Value Validation

Check every target/feature value with `np.isfinite`, and separately count NaN, positive infinity and negative infinity. Non-finite counts include NaN; do not add overlapping counts together.

In [12]:
numeric_values = sample_check[expected_columns].to_numpy(dtype=float)
finite_summary = pd.DataFrame({
    "feature": expected_columns,
    "NaN": np.isnan(numeric_values).sum(axis=0),
    "+inf": np.isposinf(numeric_values).sum(axis=0),
    "-inf": np.isneginf(numeric_values).sum(axis=0),
    "non_finite_count": (~np.isfinite(numeric_values)).sum(axis=0),
})
non_finite_rows = pd.Series((~np.isfinite(numeric_values)).any(axis=1), index=sample_check.index)
display(finite_summary)
if np.isfinite(numeric_values).all():
    print("All target and candidate feature values are finite; no NaN, +inf, or -inf detected.")
else:
    print(f"Rows containing non-finite values: {non_finite_rows.sum():,}")

,feature,NaN,+inf,-inf,non_finite_count
0,fare_amount,0,0,0,0
1,trip_distance,0,0,0,0
2,pickup_hour,0,0,0,0
3,day_of_week,0,0,0,0
4,is_weekend,0,0,0,0
5,passenger_count,0,0,0,0
6,pickup_longitude,0,0,0,0
7,pickup_latitude,0,0,0,0
8,dropoff_longitude,0,0,0,0
9,dropoff_latitude,0,0,0,0


All target and candidate feature values are finite; no NaN, +inf, or -inf detected.


## 11. Modeling Quality Summary

All counts and percentages refer to the same unfiltered prefix. Checks overlap (for example, fare > 200 is also fare > 100), so their counts must not be summed as a total number of anomalous rows. These are inspection flags, not filtering decisions.

In [13]:
quality_checks = {
    "missing rows": missing_rows,
    **fare_checks,
    **distance_checks,
    **passenger_checks,
    **temporal_checks,
    "coordinates outside cleaning bounds": coordinate_violations,
    "weekend/day-of-week inconsistency": weekend_mismatch,
    "non-finite rows": non_finite_rows,
}
quality_summary = summarize_checks(quality_checks)
display(quality_summary)

,check,count,percentage
0,missing rows,0,0.0000
1,fare <= 0,0,0.0000
2,fare < 2.5,7,0.0007
3,fare > 50,12230,1.2230
4,fare > 100,195,0.0195
5,fare > 200,12,0.0012
6,trip_distance == 0,10441,1.0441
7,trip_distance <= 0.1,16284,1.6284
8,trip_distance > 20,11543,1.1543
9,trip_distance > 30,90,0.0090


## 12. Validation Findings and Open Decisions

These findings describe the **first 1,000,000 rows** (10 numeric columns), not a random sample or the complete dataset. All required columns are present; neither the raw source header nor the loaded columns has duplicate names.

1. **Missing and finite values:** no missing cells or rows, NaN, positive infinity, or negative infinity were detected. All inspected target and feature values are finite.
2. **Passenger count:** minimum **1**, maximum **9**. Above six: **1 row (0.0001%)**, recorded as nine passengers. Below one and zero: **0 rows** each. The nine-passenger record is flagged for eligibility review, not automatically declared invalid or deleted. The 129/208 values found in Week 2 full-data analysis **do not appear in this prefix**; this does not establish their absence from the full dataset.
3. **Fare amount:** minimum **$0.01**, maximum **$500.00**. Fare > $100: **195 rows (0.0195%)**; fare > $200: **12 rows (0.0012%)**. Fare ≤ $0: **0 rows**; fare < $2.50: **7 rows (0.0007%)**. The 99th and 99.9th percentiles are **$52.00** and **$74.50**, respectively. Extreme high fares and unusually low positive fares warrant review. RMSE is sensitive to extreme target values because errors are squared; no threshold or treatment is selected here.
4. **Trip distance:** minimum **0 km**, maximum **49.484019 km**. Zero distance: **10,441 rows (1.0441%)**; distance ≤ 0.1 km (including zero): **16,284 rows (1.6284%)**. Distance > 20 km: **11,543 rows (1.1543%)**; distance > 30 km: **90 rows (0.0090%)**. Zero/near-zero distance does not by itself establish an invalid transaction; long Haversine distance alone is also not grounds for removal.
5. **Coordinates:** **0 rows** violate the inclusive Week 1 longitude bounds [−74.3, −73.7] or latitude bounds [40.5, 41.0]. All inspected coordinates remain within the cleaning bounds.
6. **Temporal features:** **0 invalid values** for each of `pickup_hour`, `day_of_week`, and `is_weekend`; all observed codes are in their respective allowed sets. Weekend/day-of-week consistency violations: **0**. These code checks pass without timezone reinterpretation; they do not independently revalidate the source datetime semantics.
7. **Open decisions:** choose a justified passenger-count range; decide how to handle observed fare extremes and zero/near-zero distances; and define a reproducible, representative final modelling-sample strategy. Schema, missing-value, finite-value, coordinate and temporal checks pass on this prefix, but must also be checked on the eventual modelling dataset. Inspection thresholds used here are not approved filtering rules.

No modelling-specific filtering rule has been applied yet.

The next step is to decide:

- reasonable passenger-count range;
- treatment of extreme fare values;
- treatment of zero/near-zero trip distance;
- final modelling sample strategy.

These decisions will be made before train/test splitting and model training.

**Stop after Phase 3A.** No final modelling dataset, split, scaling, model fitting, model evaluation, or feature importance has been produced. Phase 3B awaits review and confirmation of the handling rules.


In [14]:
print("=" * 30)
print("PHASE 3A VALIDATION COMPLETE")
print("=" * 30)
print(f"\nRows inspected:\n{rows_inspected:,}")
print(f"\nTarget:\n{TARGET}")
print(f"\nCandidate features:\n{len(FEATURES)}")
print(f"\nMissing values:\n{int(missing_counts.sum()):,} cells in {int(missing_rows.sum()):,} rows")
for label, mask in [
    ("Passenger count > 6", passenger_checks["passenger_count > 6"]),
    ("Fare > 100", fare_checks["fare > 100"]),
    ("Zero-distance trips", distance_checks["trip_distance == 0"]),
    ("Distance > 20 km", distance_checks["trip_distance > 20"]),
    ("Coordinate violations", coordinate_violations),
    ("Temporal violations", temporal_violations),
]:
    print(f"\n{label}:\n{int(mask.sum()):,} ({mask.mean() * 100:.6f}%)")
print("\nNext step:\nDefine modelling-specific eligibility rules before sampling and train/test splitting.")

PHASE 3A VALIDATION COMPLETE

Rows inspected:
1,000,000

Target:
fare_amount

Candidate features:
9

Missing values:
0 cells in 0 rows

Passenger count > 6:
1 (0.000100%)

Fare > 100:
195 (0.019500%)

Zero-distance trips:
10,441 (1.044100%)

Distance > 20 km:
11,543 (1.154300%)

Coordinate violations:
0 (0.000000%)

Temporal violations:
0 (0.000000%)

Next step:
Define modelling-specific eligibility rules before sampling and train/test splitting.


# Phase 3B — Modeling Dataset Preparation

The modelling-specific eligibility rules are deliberately conservative. Passenger counts outside 1–6 and fares below the modelling minimum of $2.50 are excluded from the primary regression population; all required target/feature values must also be finite and non-missing. Counts above six are not treated as normal numeric party sizes for this population; this does not mean every such trip is impossible.

Zero-distance, near-zero-distance, long-distance and high-fare trips are retained because there is insufficient evidence to classify all such records as invalid. Their predictive behaviour will instead be evaluated during residual and subgroup analysis. **There is no maximum fare threshold or distance cutoff.** This policy supersedes the previous Phase 3B draft; Phase 3A remains an unchanged historical validation record.

This phase performs a full eligibility audit, uniform random sampling of exactly one million eligible records, sample sanity checks, an 80/20 train/test split, and metadata export. No model, scaler, predictions, feature selection or evaluation metrics are produced.

## 13. Full-Dataset Eligibility Audit

Read only the target plus the nine existing features in 500,000-row chunks. One complete pass both counts the full source population and constructs the sample, avoiding another scan. Audit flags overlap and their counts must not be summed as exclusions. Eligibility is the joint fare/passenger/finite/non-missing condition only; coordinate or temporal violations are reported and stop preparation for review, without silently adding filters.

In [3]:
import json
from datetime import datetime, timezone
import sklearn
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
MODEL_SAMPLE_SIZE = 1_000_000
CHUNK_SIZE = 500_000
MIN_FARE = 2.50
MIN_PASSENGERS = 1
MAX_PASSENGERS = 6
TEST_SIZE = 0.20
NEAR_ZERO_KM = 0.1  # Audit only.
LONG_DISTANCE_THRESHOLDS = (20, 30)  # Audit only.
HIGH_FARE_THRESHOLDS = (100, 200)  # Audit only.
TARGET = "fare_amount"
FEATURES = [
    "trip_distance", "pickup_hour", "day_of_week", "is_weekend",
    "passenger_count", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
]
REQUIRED_COLUMNS = [TARGET] + FEATURES
COORDINATE_BOUNDS = {
    "pickup_longitude": (-74.3, -73.7), "dropoff_longitude": (-74.3, -73.7),
    "pickup_latitude": (40.5, 41.0), "dropoff_latitude": (40.5, 41.0),
}
TEMPORAL_ALLOWED = {"pickup_hour": list(range(24)), "day_of_week": list(range(7)), "is_weekend": [0, 1]}
REPORT_DIR = Path("../reports")
SAMPLE_PATH = Path("../data/processed/modeling_sample_1m.csv")
AUDIT_PATH = REPORT_DIR / "modeling_eligibility_audit.csv"
ELIGIBILITY_METADATA_PATH = REPORT_DIR / "modeling_eligibility_metadata.json"
SPLIT_METADATA_PATH = REPORT_DIR / "modeling_split_metadata.json"
ELIGIBILITY_RULES = {
    "passenger_count": {"minimum_inclusive": MIN_PASSENGERS, "maximum_inclusive": MAX_PASSENGERS},
    "fare_amount": {"minimum_inclusive": MIN_FARE, "maximum": None},
    "required_values": "All target and feature values must be finite and non-missing",
    "trip_distance": "No distance-based exclusion; zero, near-zero and long trips retained",
    "coordinates_and_temporal": "Audit only; stop for review on violations, no silent filtering",
}
SAMPLING_METHOD = "One-pass random-priority reservoir over eligible rows; no stratification"
assert TARGET not in FEATURES
assert not {"key", "pickup_datetime"}.intersection(FEATURES)
assert SAMPLE_PATH.resolve() != DATA_PATH.resolve()
# The existing root ignore rule covers this local derived dataset; do not stage it.
ignore_rules = Path("../.gitignore").read_text().splitlines()
assert "data/processed/" in [line.strip() for line in ignore_rules]
print("Confirmed existing .gitignore exclusion: data/processed/")
print("Target:", TARGET, "| Features:", FEATURES)
print("Eligibility policy:")
print(json.dumps(ELIGIBILITY_RULES, indent=2))

Confirmed existing .gitignore exclusion: data/processed/
Target: fare_amount | Features: ['trip_distance', 'pickup_hour', 'day_of_week', 'is_weekend', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
Eligibility policy:
{
  "passenger_count": {
    "minimum_inclusive": 1,
    "maximum_inclusive": 6
  },
  "fare_amount": {
    "minimum_inclusive": 2.5,
    "maximum": null
  },
  "required_values": "All target and feature values must be finite and non-missing",
  "trip_distance": "No distance-based exclusion; zero, near-zero and long trips retained",
  "coordinates_and_temporal": "Audit only; stop for review on violations, no silent filtering"
}


### Uniform Sampling Method and Reproducibility

A first-N prefix can reflect source-file order. Assign every source row an independent pseudo-random uniform priority with one `numpy.default_rng(42)` generator, then apply eligibility and retain the one million eligible rows with the smallest priorities across **all** chunks. Priorities are generated for every source row and do not depend on fare, distance, time or location. Conditional on eligibility, each record has the same inclusion opportunity, without stratification. With ideal continuous priorities this is uniform sampling without replacement; pseudo-random finite-precision priorities approximate that construction.

The reservoir retains at most one million rows plus one input chunk and temporary selection arrays. Sorting selected rows by original source position makes split input order deterministic; these temporary source indices are never exported. The derived sample gets its own unique RangeIndex before splitting and is saved with `index=False`. Reproducibility depends on unchanged source content/order, seed, policy and the recorded package versions. If fewer than one million eligible rows exist, preparation stops rather than silently returning a smaller final sample.

Exact full-population hour/day counts are accumulated during the same scan for basic representativeness checks. Full-data exact quantiles are not computed.

In [4]:
def audit_flags(frame):
    numeric = frame[REQUIRED_COLUMNS].to_numpy(dtype=float)
    missing = frame[REQUIRED_COLUMNS].isna().any(axis=1)
    nonfinite = pd.Series(~np.isfinite(numeric).all(axis=1), index=frame.index)
    coordinate_bad = pd.Series(False, index=frame.index)
    for column, (lower, upper) in COORDINATE_BOUNDS.items():
        coordinate_bad |= (frame[column] < lower) | (frame[column] > upper)
    temporal_bad = pd.Series(False, index=frame.index)
    for column, allowed in TEMPORAL_ALLOWED.items():
        temporal_bad |= ~frame[column].isin(allowed)
    valid_weekly = frame["day_of_week"].isin(TEMPORAL_ALLOWED["day_of_week"]) & frame["is_weekend"].isin([0, 1])
    flags = {
        "fare_below_2_5": frame[TARGET] < MIN_FARE,
        "passenger_below_1": frame["passenger_count"] < MIN_PASSENGERS,
        "passenger_above_6": frame["passenger_count"] > MAX_PASSENGERS,
        "zero_distance": frame["trip_distance"] == 0,
        "near_zero_distance_le_0_1": frame["trip_distance"] <= NEAR_ZERO_KM,
        "missing_rows": missing,
        "nonfinite_rows": nonfinite,  # Includes missing numeric values.
        "coordinate_violations": coordinate_bad,
        "temporal_violations": temporal_bad,
        "weekend_consistency_violations": valid_weekly & (
            frame["is_weekend"] != (frame["day_of_week"] >= 5).astype(int)
        ),
    }
    for threshold in LONG_DISTANCE_THRESHOLDS:
        flags[f"distance_above_{threshold}"] = frame["trip_distance"] > threshold
    for threshold in HIGH_FARE_THRESHOLDS:
        flags[f"fare_above_{threshold}"] = frame[TARGET] > threshold
    eligible = (
        frame[TARGET].ge(MIN_FARE)
        & frame["passenger_count"].between(MIN_PASSENGERS, MAX_PASSENGERS)
        & ~nonfinite & ~missing
    )
    return flags, eligible


def audit_and_sample(path, target_size, chunksize, random_state):
    if target_size <= 0 or chunksize <= 0:
        raise ValueError("Sample and chunk sizes must be positive")
    rng = np.random.default_rng(random_state)
    totals = {"total_rows": 0, "eligible_rows": 0}
    reservoir = None
    priorities = np.empty(0, dtype=np.float64)
    full_group_counts = {c: pd.Series(dtype="int64") for c in ["pickup_hour", "day_of_week"]}
    eligible_group_counts = {c: pd.Series(dtype="int64") for c in full_group_counts}
    with pd.read_csv(path, usecols=REQUIRED_COLUMNS, chunksize=chunksize) as reader:
        for chunk_number, chunk in enumerate(reader, start=1):
            non_numeric = chunk.select_dtypes(exclude="number").columns.tolist()
            if non_numeric:
                raise ValueError(f"Non-numeric fields in chunk {chunk_number}: {non_numeric}")
            chunk.index = pd.RangeIndex(totals["total_rows"], totals["total_rows"] + len(chunk), name="source_row_position")
            totals["total_rows"] += len(chunk)
            flags, eligible = audit_flags(chunk)
            for name, mask in flags.items():
                totals[name] = totals.get(name, 0) + int(mask.sum())
            totals["eligible_rows"] += int(eligible.sum())
            for column in full_group_counts:
                full_group_counts[column] = full_group_counts[column].add(chunk[column].value_counts(), fill_value=0)
                eligible_group_counts[column] = eligible_group_counts[column].add(chunk.loc[eligible, column].value_counts(), fill_value=0)
            # Generate priorities for all source rows before selecting eligible candidates.
            chunk_priorities = rng.random(len(chunk))[eligible.to_numpy()]
            candidates = chunk.loc[eligible, REQUIRED_COLUMNS]
            if candidates.empty:
                continue
            if reservoir is None:
                reservoir = candidates
                priorities = chunk_priorities
                if len(reservoir) > target_size:
                    keep = np.argpartition(priorities, target_size - 1)[:target_size]
                    reservoir = reservoir.iloc[keep]
                    priorities = priorities[keep]
            else:
                old_rows = len(reservoir)
                combined = np.concatenate((priorities, chunk_priorities))
                if len(combined) > target_size:
                    keep = np.argpartition(combined, target_size - 1)[:target_size]
                    keep_old = keep[keep < old_rows]
                    keep_new = keep[keep >= old_rows] - old_rows
                    selected_priorities = np.concatenate((priorities[keep_old], chunk_priorities[keep_new]))
                    reservoir = pd.concat((reservoir.iloc[keep_old], candidates.iloc[keep_new]))
                    priorities = selected_priorities
                else:
                    reservoir = pd.concat((reservoir, candidates))
                    priorities = combined
            if chunk_number % 20 == 0:
                print(f"Audited {totals['total_rows']:,} source rows; {totals['eligible_rows']:,} eligible", flush=True)
    totals["excluded_rows"] = totals["total_rows"] - totals["eligible_rows"]
    if reservoir is None:
        reservoir = pd.DataFrame(columns=REQUIRED_COLUMNS)
    reservoir = reservoir.sort_index()
    assert reservoir.index.is_unique
    assert len(reservoir) == min(target_size, totals["eligible_rows"])
    return reservoir, totals, full_group_counts, eligible_group_counts

source_stat_before = DATA_PATH.stat()
modeling_sample, full_audit_counts, full_group_counts, eligible_group_counts = audit_and_sample(
    DATA_PATH, MODEL_SAMPLE_SIZE, CHUNK_SIZE, RANDOM_STATE,
)
source_stat_after = DATA_PATH.stat()
assert (source_stat_before.st_size, source_stat_before.st_mtime_ns) == (
    source_stat_after.st_size, source_stat_after.st_mtime_ns
), "Source changed during audit"
full_rows = full_audit_counts["total_rows"]
eligible_rows = full_audit_counts["eligible_rows"]
excluded_rows = full_audit_counts["excluded_rows"]
full_audit = pd.DataFrame([
    {"check": name, "count": count, "percentage": count / full_rows * 100}
    for name, count in full_audit_counts.items()
])
display(full_audit)
assert eligible_rows + excluded_rows == full_rows
full_audit.to_csv(AUDIT_PATH, index=False)
created_at_utc = datetime.now(timezone.utc).isoformat()
eligibility_metadata = {
    "source_dataset": str(DATA_PATH.resolve()),
    "source_size_bytes": source_stat_before.st_size,
    "source_mtime_ns": source_stat_before.st_mtime_ns,
    "full_row_count": full_rows, "eligible_row_count": eligible_rows,
    "excluded_row_count": excluded_rows,
    "eligibility_rules": ELIGIBILITY_RULES, "chunk_size": CHUNK_SIZE,
    "random_seed": RANDOM_STATE, "created_at_utc": created_at_utc,
    "target": TARGET, "features": FEATURES,
    "sampling_method": SAMPLING_METHOD, "requested_sample_rows": MODEL_SAMPLE_SIZE,
    "actual_sample_rows": len(modeling_sample),
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
    "audit_note": "Flag counts overlap; percentages use full source rows; nonfinite includes NaN",
    "audit_counts": full_audit_counts,
}
ELIGIBILITY_METADATA_PATH.write_text(json.dumps(eligibility_metadata, indent=2) + "\n")
print(f"Saved full audit: {AUDIT_PATH}")
print(f"Saved eligibility metadata: {ELIGIBILITY_METADATA_PATH}")
print(f"Uniform sample requested / obtained: {MODEL_SAMPLE_SIZE:,} / {len(modeling_sample):,}; seed={RANDOM_STATE}")
# Audit reports are saved even if an unexpected violation requires stopping for review.
for check in ["coordinate_violations", "temporal_violations", "weekend_consistency_violations"]:
    if full_audit_counts[check]:
        raise ValueError(f"Full audit found {full_audit_counts[check]:,} {check}; review required, no silent filtering")
if eligible_rows < MODEL_SAMPLE_SIZE:
    raise ValueError(f"Only {eligible_rows:,} eligible rows; cannot construct the requested exact sample")

Audited 10,000,000 source rows; 9,999,870 eligible
Audited 20,000,000 source rows; 19,999,734 eligible
Audited 30,000,000 source rows; 29,999,603 eligible
Audited 40,000,000 source rows; 39,999,473 eligible
Audited 50,000,000 source rows; 49,999,322 eligible
Saved full audit: ../reports/modeling_eligibility_audit.csv
Saved eligibility metadata: ../reports/modeling_eligibility_metadata.json
Uniform sample requested / obtained: 1,000,000 / 1,000,000; seed=42


,check,count,percentage
0,total_rows,54004358,100.000000
1,eligible_rows,54003614,99.998622
2,fare_below_2_5,680,0.001259
3,passenger_below_1,0,0.000000
4,passenger_above_6,64,0.000119
5,zero_distance,557212,1.031791
6,near_zero_distance_le_0_1,870955,1.612749
7,missing_rows,0,0.000000
8,nonfinite_rows,0,0.000000
9,coordinate_violations,0,0.000000


## 14. Modeling Sample Validation

Validate exactly one million rows and ten required columns. High fares and zero/near-zero/long distances are audit subgroups, not exclusion rules. Display their actual counts, along with fare/distance quantiles and passenger/hour/day distributions. Reset only the in-memory sample index; the CSV contains no identifier column and is excluded from Git by the existing `data/processed/` rule.

In [5]:
modeling_sample = modeling_sample[REQUIRED_COLUMNS].reset_index(drop=True)
assert modeling_sample.shape == (MODEL_SAMPLE_SIZE, len(REQUIRED_COLUMNS))
assert len(REQUIRED_COLUMNS) == 10
assert list(modeling_sample.columns) == REQUIRED_COLUMNS
sample_missing_values = int(modeling_sample.isna().sum().sum())
sample_numeric = modeling_sample.to_numpy(dtype=float)
sample_nonfinite_values = int((~np.isfinite(sample_numeric)).sum())
assert sample_missing_values == 0 and sample_nonfinite_values == 0
assert modeling_sample["passenger_count"].between(MIN_PASSENGERS, MAX_PASSENGERS).all()
assert modeling_sample[TARGET].ge(MIN_FARE).all()
del sample_numeric
sample_flags, sample_eligible = audit_flags(modeling_sample)
assert sample_eligible.all()
sample_subgroup_summary = pd.DataFrame([
    {"check": name, "count": int(mask.sum()), "percentage": mask.mean() * 100}
    for name, mask in sample_flags.items()
])
print("Modeling sample shape:", modeling_sample.shape)
print(f"Missing values: {sample_missing_values}; non-finite values: {sample_nonfinite_values}")
display(modeling_sample[[TARGET, "trip_distance", "passenger_count"]].agg(["min", "max"]).T)
display(sample_subgroup_summary)
QUANTILE_LEVELS = [0, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999, 1]
sample_fare_quantiles = modeling_sample[TARGET].quantile(QUANTILE_LEVELS)
sample_distance_quantiles = modeling_sample["trip_distance"].quantile(QUANTILE_LEVELS)
print("Sample fare quantiles:")
display(sample_fare_quantiles)
print("Sample distance quantiles (km):")
display(sample_distance_quantiles)
for column in ["passenger_count", "pickup_hour", "day_of_week"]:
    counts = modeling_sample[column].value_counts().sort_index()
    print(f"Sample {column} distribution:")
    display(pd.DataFrame({"count": counts, "percentage": counts / len(modeling_sample) * 100}))
# Derived local sample only; no writes to train_features.csv or train_clean.csv.
modeling_sample.to_csv(SAMPLE_PATH, index=False)
print(f"Saved local derived sample: {SAMPLE_PATH.resolve()}")
print("The CSV contains the ten required columns only; no row indices were saved or staged.")

Modeling sample shape: (1000000, 10)
Missing values: 0; non-finite values: 0
Sample fare quantiles:
Sample distance quantiles (km):
Sample passenger_count distribution:
Sample pickup_hour distribution:
Sample day_of_week distribution:
Saved local derived sample: /Users/xiaochuan/nyc-taxi-fare-prediction/data/processed/modeling_sample_1m.csv
The CSV contains the ten required columns only; no row indices were saved or staged.


,min,max
fare_amount,2.5,500.000000
trip_distance,0.0,45.504764
passenger_count,1.0,6.000000


,check,count,percentage
0,fare_below_2_5,0,0.0000
1,passenger_below_1,0,0.0000
2,passenger_above_6,0,0.0000
3,zero_distance,10449,1.0449
4,near_zero_distance_le_0_1,16201,1.6201
5,missing_rows,0,0.0000
6,nonfinite_rows,0,0.0000
7,coordinate_violations,0,0.0000
8,temporal_violations,0,0.0000
9,weekend_consistency_violations,0,0.0000


0.000      2.5000
0.010      3.3000
0.050      4.1000
0.500      8.5000
0.950     30.0000
0.990     52.0000
0.999     74.5001
1.000    500.0000
Name: fare_amount, dtype: float64

0.000     0.000000
0.010     0.000000
0.050     0.559092
0.500     2.151932
0.950     9.950730
0.990    20.278693
0.999    23.050337
1.000    45.504764
Name: trip_distance, dtype: float64

,count,percentage
passenger_count,,
1,693854,69.3854
2,148312,14.8312
3,44165,4.4165
4,21437,2.1437
5,71107,7.1107
6,21125,2.1125


,count,percentage
pickup_hour,,
0,39858,3.9858
1,29386,2.9386
2,22000,2.2000
3,15958,1.5958
4,11744,1.1744
5,9885,0.9885
6,20443,2.0443
7,35809,3.5809
8,45431,4.5431


,count,percentage
day_of_week,,
0,128297,12.8297
1,140183,14.0183
2,144910,14.4910
3,149369,14.9369
4,153692,15.3692
5,151934,15.1934
6,131615,13.1615


## 15. Sample Representativeness

Compare sample fare/distance quantiles with the supplied approximate Week 2 references and the observed Phase 3A prefix quantiles. References describe earlier data, not exact quantiles of the newly eligible population; small differences are expected from eligibility and sampling. These are descriptive sanity checks with **no mechanical pass/fail threshold**.

For hours and days, compare the sample with exact **full source** and **eligible population** shares collected during the same audit pass. Percentage-point differences from the eligible population are the most directly comparable measure. No sampling adjustments are made based on these comparisons.

In [6]:
REFERENCE_QUANTILES = {
    ("fare_amount", 0.50): 8.50, ("fare_amount", 0.95): 30.27, ("fare_amount", 0.99): 52.50,
    ("trip_distance", 0.50): 2.15, ("trip_distance", 0.95): 10.02, ("trip_distance", 0.99): 20.31,
}
quantile_records = []
for (feature, quantile), reference in REFERENCE_QUANTILES.items():
    phase3a_quantiles = fare_quantiles if feature == TARGET else distance_quantiles
    sample_quantiles = sample_fare_quantiles if feature == TARGET else sample_distance_quantiles
    sample_value = sample_quantiles.loc[quantile]
    quantile_records.append({
        "feature": feature, "quantile": quantile,
        "week2_approx_reference": reference,
        "phase3a_prefix": phase3a_quantiles.loc[quantile],
        "modeling_sample": sample_value,
        "sample_minus_week2": sample_value - reference,
    })
quantile_comparison = pd.DataFrame(quantile_records)
display(quantile_comparison)
group_comparisons = {}
for column in ["pickup_hour", "day_of_week"]:
    categories = TEMPORAL_ALLOWED[column]
    full_counts = full_group_counts[column].reindex(categories, fill_value=0)
    eligible_counts = eligible_group_counts[column].reindex(categories, fill_value=0)
    sample_counts = modeling_sample[column].value_counts().reindex(categories, fill_value=0)
    assert int(full_counts.sum()) == full_rows
    assert int(eligible_counts.sum()) == eligible_rows
    comparison = pd.DataFrame({
        "full_source_pct": full_counts / full_rows * 100,
        "eligible_population_pct": eligible_counts / eligible_rows * 100,
        "modeling_sample_pct": sample_counts / len(modeling_sample) * 100,
    })
    comparison.index.name = column
    comparison["sample_minus_eligible_pp"] = comparison["modeling_sample_pct"] - comparison["eligible_population_pct"]
    group_comparisons[column] = comparison
    print(f"{column} distribution comparison:")
    display(comparison)
    print(f"Maximum absolute share difference: {comparison['sample_minus_eligible_pp'].abs().max():.4f} percentage points")

pickup_hour distribution comparison:
Maximum absolute share difference: 0.0446 percentage points
day_of_week distribution comparison:
Maximum absolute share difference: 0.0439 percentage points


,feature,quantile,week2_approx_reference,phase3a_prefix,modeling_sample,sample_minus_week2
0,fare_amount,0.50,8.50,8.500000,8.500000,0.000000
1,fare_amount,0.95,30.27,30.100000,30.000000,-0.270000
2,fare_amount,0.99,52.50,52.000000,52.000000,-0.500000
3,trip_distance,0.50,2.15,2.153391,2.151932,0.001932
4,trip_distance,0.95,10.02,9.989398,9.950730,-0.069270
5,trip_distance,0.99,20.31,20.263769,20.278693,-0.031307


,full_source_pct,eligible_population_pct,modeling_sample_pct,sample_minus_eligible_pp
pickup_hour,,,,
0,3.958875,3.958879,3.9858,0.026921
1,2.932154,2.932106,2.9386,0.006494
2,2.186512,2.186489,2.2000,0.013511
3,1.602073,1.602056,1.5958,-0.006256
4,1.160704,1.160665,1.1744,0.013735
5,0.977475,0.977462,0.9885,0.011038
6,2.051640,2.051640,2.0443,-0.007340
7,3.601315,3.601337,3.5809,-0.020437
8,4.542757,4.542783,4.5431,0.000317


,full_source_pct,eligible_population_pct,modeling_sample_pct,sample_minus_eligible_pp
day_of_week,,,,
0,12.818116,12.818100,12.8297,0.011600
1,14.009006,14.008992,14.0183,0.009308
2,14.507613,14.507644,14.4910,-0.016644
3,14.980798,14.980807,14.9369,-0.043907
4,15.385162,15.385161,15.3692,-0.015961
5,15.179671,15.179641,15.1934,0.013759
6,13.119634,13.119655,13.1615,0.041845


## 16. Train/Test Split

Use the same nine existing features and `fare_amount` as target. Split 80% training / 20% test with seed 42 and no stratification. No scaling or model-specific preprocessing is applied. The test partition is reserved for final evaluation after future model decisions are fixed; the checks below assess alignment and broad distribution consistency only. Random historical row splitting does not by itself validate future-period forecasting performance.

In [7]:
X = modeling_sample[FEATURES]
y = modeling_sample[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
print(f"Train rows: {len(X_train):,} ({len(X_train) / len(X):.2%})")
print(f"Test rows: {len(X_test):,} ({len(X_test) / len(X):.2%})")

Train rows: 800,000 (80.00%)
Test rows: 200,000 (20.00%)


## 17. Leakage and Split Validation

The target, record identifier and raw datetime string are absent from X. Preserve the sample's unique in-memory row index through splitting to check disjoint partitions and X/y alignment. No index lists or large partition CSVs are exported. These checks establish column/row separation, not an exhaustive claim that every future modelling pipeline is leakage-free.

In [8]:
assert not {TARGET, "key", "pickup_datetime"}.intersection(X.columns)
assert list(X.columns) == FEATURES
assert X.index.is_unique
assert len(X_train) + len(X_test) == len(modeling_sample)
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert y_train.equals(y.loc[X_train.index])
assert y_test.equals(y.loc[X_test.index])
assert not y_train.isna().any() and not y_test.isna().any()
split_overlap = len(X_train.index.intersection(X_test.index))
assert split_overlap == 0
combined_indices = X_train.index.append(X_test.index)
assert combined_indices.is_unique
assert combined_indices.sort_values().equals(X.index.sort_values())
del combined_indices
split_target_summary = pd.DataFrame([
    {"split": name, "rows": len(target), "mean": target.mean(), "median": target.median()}
    for name, target in [("train", y_train), ("test", y_test)]
]).set_index("split")
display(split_target_summary)
print(f"Train/test overlap: {split_overlap}")
print("All sample and split assertions passed.")
split_metadata = {
    "source_dataset": str(DATA_PATH.resolve()),
    "modelling_sample_path": str(SAMPLE_PATH.resolve()),
    "modelling_sample_size": len(modeling_sample),
    "eligible_population_size": eligible_rows,
    "features": FEATURES, "target": TARGET,
    "random_state": RANDOM_STATE, "test_size": TEST_SIZE,
    "train_rows": len(X_train), "test_rows": len(X_test),
    "eligibility_rules": ELIGIBILITY_RULES,
    "sampling_method": SAMPLING_METHOD,
    "versions": eligibility_metadata["versions"],
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "split_overlap": split_overlap,
    "target_summary": split_target_summary.reset_index().to_dict(orient="records"),
    "sample_subgroup_counts": {name: int(mask.sum()) for name, mask in sample_flags.items()},
    "sample_fare_quantiles": {str(q): float(value) for q, value in sample_fare_quantiles.items()},
    "sample_distance_quantiles": {str(q): float(value) for q, value in sample_distance_quantiles.items()},
    "sample_hour_distribution_pct": group_comparisons["pickup_hour"]["modeling_sample_pct"].to_dict(),
    "sample_day_distribution_pct": group_comparisons["day_of_week"]["modeling_sample_pct"].to_dict(),
    "index_note": "Unique sample positions used only in memory; no indices or partition CSVs exported",
}
SPLIT_METADATA_PATH.write_text(json.dumps(split_metadata, indent=2) + "\n")
print(f"Saved split metadata: {SPLIT_METADATA_PATH}")

Train/test overlap: 0
All sample and split assertions passed.
Saved split metadata: ../reports/modeling_split_metadata.json


,rows,mean,median
split,,,
train,800000,11.268651,8.5
test,200000,11.264545,8.5


### Phase 3B Findings and Stop Point

- The full audit inspected **54,004,358 rows**, of which **54,003,614** are eligible and **744** are excluded. There are **680** fares below $2.50 and **64** passenger counts above six, with no below-one counts. In this run these exclusions do not overlap. Missing, non-finite, coordinate, temporal and weekend-consistency violation counts are all **zero**.
- The source includes **557,212 zero-distance trips**, **870,955 trips ≤0.1 km**, **633,221 trips >20 km**, **10,505 fares >$100** and **427 fares >$200**. None of those subgroup flags independently makes a row ineligible.
- The uniform eligible sample has shape **(1,000,000, 10)**, passenger range **1–6**, fare range **$2.50–$500.00**, and distance range **0.000000–45.504764 km**. It retains **10,449 zero-distance**, **16,201 near-zero (including zero)**, **11,580 >20 km**, **161 >$100** and **6 >$200** records. No maximum fare or distance cutoff was applied.
- Sample fare median/P95/P99 are **$8.50/$30.00/$52.00**; distance median/P95/P99 are **2.151932/9.950730/20.278693 km**. These are broadly close to the supplied approximate Week 2 references. The largest absolute share differences from the exact eligible population are **0.0446 percentage points** by hour and **0.0439 percentage points** by day. These basic comparisons show no obvious distribution shift, without proving equivalence or triggering resampling.
- The split contains **800,000 training** and **200,000 test** rows, with **0 overlapping indices**. Train/test fare means are **$11.268651/$11.264545**; both medians are **$8.50**. All sample and split assertions pass, and no additional data issue was detected by the specified audit.
- The audit and two metadata JSON files are saved under `reports/`. The local derived sample is `data/processed/modeling_sample_1m.csv`, covered by the existing Git ignore rule. No row index lists or train/test CSVs are exported; source datasets and earlier notebooks remain unchanged.

**Stop here.** Phase 3C requires manual review. No regression model, prediction, scaler, evaluation metric or feature importance has been computed.


In [9]:
print("=" * 36)
print("PHASE 3B DATA PREPARATION COMPLETE")
print("=" * 36)
for label, value in [
    ("Full source rows", full_rows), ("Eligible rows", eligible_rows),
    ("Excluded rows", excluded_rows), ("Final modelling sample", len(modeling_sample)),
    ("Train rows", len(X_train)), ("Test rows", len(X_test)),
]:
    print(f"\n{label}:\n{value:,}")
print(f"\nTarget:\n{TARGET}")
print(f"\nFeatures:\n{len(FEATURES)}")
print(f"\nPassenger range:\n{modeling_sample['passenger_count'].min()}–{modeling_sample['passenger_count'].max()}")
print(f"\nFare range:\n${modeling_sample[TARGET].min():.2f}–${modeling_sample[TARGET].max():.2f}")
print(f"\nZero-distance rows retained:\n{int(sample_flags['zero_distance'].sum()):,}")
print(f"\nHigh-fare (>100) rows retained:\n{int(sample_flags['fare_above_100'].sum()):,}")
print(f"\nTrain/Test overlap:\n{split_overlap}")
print("\nNext step:\nPhase 3C — Baseline Regression")
print("STOP: No model trained; wait for manual review before Phase 3C.")

PHASE 3B DATA PREPARATION COMPLETE

Full source rows:
54,004,358

Eligible rows:
54,003,614

Excluded rows:
744

Final modelling sample:
1,000,000

Train rows:
800,000

Test rows:
200,000

Target:
fare_amount

Features:
9

Passenger range:
1–6

Fare range:
$2.50–$500.00

Zero-distance rows retained:
10,449

High-fare (>100) rows retained:
161

Train/Test overlap:
0

Next step:
Phase 3C — Baseline Regression
STOP: No model trained; wait for manual review before Phase 3C.


## Phase 3B — Predictive Modeling

This addition defines and checks the initial modelling inputs and reviews the frozen train/test split. All earlier preparation cells, outputs and section labels are preserved. No model training, predictions, scoring, preprocessing or new split is performed.

### 1. Feature and Target Definition

`fare_amount` is the sole target. The initial features are `trip_distance`, `pickup_hour`, `day_of_week`, `is_weekend`, `passenger_count`, `pickup_longitude`, `pickup_latitude`, `dropoff_longitude`, and `dropoff_latitude`.

None is directly constructed from `fare_amount`, so their definitions show no obvious target leakage. Distance and coordinates contain overlapping information, as do `day_of_week` and `is_weekend`; redundancy is not target leakage. Keep all nine for now rather than manually discard potentially useful information. Their incremental value may be assessed through model performance and feature importance in a later, approved stage.

In [1]:
# Reuse earlier imports, sample path and DataFrame when available.
# The fallback supports running only this addition after a kernel restart.
if "Path" not in globals():
    from pathlib import Path
if "np" not in globals():
    import numpy as np
if "pd" not in globals():
    import pandas as pd

TARGET = "fare_amount"
FEATURES = [
    "trip_distance",
    "pickup_hour",
    "day_of_week",
    "is_weekend",
    "passenger_count",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
]
required_modeling_columns = [TARGET] + FEATURES
assert TARGET not in FEATURES
assert len(FEATURES) == len(set(FEATURES)) == 9
if "SAMPLE_PATH" not in globals():
    SAMPLE_PATH = Path("../data/processed/modeling_sample_1m.csv")
# Existing convention is notebooks/ as cwd. Adapt the same variable when cwd is root.
if not SAMPLE_PATH.is_file() and Path("data/processed/modeling_sample_1m.csv").is_file():
    SAMPLE_PATH = Path("data/processed/modeling_sample_1m.csv")
if "modeling_sample" in globals() and isinstance(modeling_sample, pd.DataFrame):
    modeling_df = modeling_sample
    dataset_reuse_status = "Reused existing modeling_sample DataFrame"
elif "modeling_df" in globals() and isinstance(modeling_df, pd.DataFrame):
    dataset_reuse_status = "Reused existing modeling_df DataFrame"
else:
    if not SAMPLE_PATH.is_file():
        raise FileNotFoundError(f"Saved modeling sample not found: {SAMPLE_PATH.resolve()}")
    modeling_df = pd.read_csv(SAMPLE_PATH)
    dataset_reuse_status = "Loaded the existing saved modeling sample; no resampling"
missing_required = sorted(set(required_modeling_columns) - set(modeling_df.columns))
assert not missing_required, f"Missing required columns: {missing_required}"
print("Target:", TARGET)
print("Feature list:", FEATURES)
print(dataset_reuse_status)
print("Existing sample path:", SAMPLE_PATH.resolve())

Target: fare_amount
Feature list: ['trip_distance', 'pickup_hour', 'day_of_week', 'is_weekend', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
Loaded the existing saved modeling sample; no resampling
Existing sample path: /Users/xiaochuan/nyc-taxi-fare-prediction/data/processed/modeling_sample_1m.csv


### 2. Modeling Dataset Integrity Check

Verify the approved million-row population and ten numeric columns, with finite/non-missing values, minimum fare $2.50, passenger counts 1–6 and valid temporal codes. Do not filter, clip or transform any values. Zero-distance and high-fare records retain their previously approved treatment.

In [2]:
assert modeling_df.shape == (1_000_000, 10)
assert modeling_df.columns.is_unique
assert set(modeling_df.columns) == set(required_modeling_columns)
assert all(pd.api.types.is_numeric_dtype(modeling_df[column]) for column in required_modeling_columns)
modeling_missing_count = int(modeling_df.isna().sum().sum())
all_modeling_values_finite = bool(np.isfinite(modeling_df.to_numpy()).all())
assert modeling_missing_count == 0
assert all_modeling_values_finite
assert modeling_df[TARGET].ge(2.50).all()
assert modeling_df["passenger_count"].between(1, 6).all()
assert modeling_df["pickup_hour"].isin(range(24)).all()
assert modeling_df["day_of_week"].isin(range(7)).all()
assert modeling_df["is_weekend"].isin([0, 1]).all()
assert modeling_df.index.is_unique
retained_zero_distance_rows = int(modeling_df["trip_distance"].eq(0).sum())
retained_high_fare_rows = int(modeling_df[TARGET].gt(100).sum())
print("Modeling dataset shape:", modeling_df.shape)
print("Missing value count:", modeling_missing_count)
print("All values finite:", all_modeling_values_finite)
print(f"Zero-distance rows retained: {retained_zero_distance_rows:,}")
print(f"Fare > $100 rows retained: {retained_high_fare_rows:,}")
print("Target descriptive statistics:")
display(modeling_df[TARGET].describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]))
print("Feature dtypes:")
display(modeling_df[FEATURES].dtypes)

Modeling dataset shape: (1000000, 10)
Missing value count: 0
All values finite: True
Zero-distance rows retained: 10,449
Fare > $100 rows retained: 161
Target descriptive statistics:
Feature dtypes:


count    1000000.000000
mean          11.267829
std            9.439199
min            2.500000
25%            6.000000
50%            8.500000
75%           12.500000
95%           30.000000
99%           52.000000
max          500.000000
Name: fare_amount, dtype: float64

trip_distance        float64
pickup_hour            int64
day_of_week            int64
is_weekend             int64
passenger_count        int64
pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
dtype: object

### 3. X / y Definition

Copy the nine feature columns into X and the sole target into y. Preserve row order, index and values. No scaling, encoding, clipping or feature selection is applied.

In [3]:
X = modeling_df[FEATURES].copy()
y = modeling_df[TARGET].copy()
assert X.shape == (1_000_000, 9)
assert y.shape == (1_000_000,)
assert TARGET not in X.columns
assert y.name == TARGET
assert X.index.equals(y.index)
assert X.equals(modeling_df[FEATURES])
assert y.equals(modeling_df[TARGET])
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000000, 9)
y shape: (1000000,)


### 4. Existing Train / Test Split

Use the reviewed preparation split for all future model comparisons; do not create a second split. The earlier code uses `train_test_split(X, y, test_size=0.20, random_state=42)`, with default shuffling and no stratification, giving 800,000 training and 200,000 test rows. The saved sample is in deterministic source-position order before its index is reset.

`reports/modeling_split_metadata.json` stores the sample path and size, features, target, seed, test fraction, row counts, library versions, target summaries and recorded zero overlap. It explicitly records that row indices and partition CSVs were **not exported**. In a continuing kernel, reuse `X_train`, `X_test`, `y_train`, `y_test` and check them against X/y.

With a fresh kernel, those objects are absent. The original split algorithm is deterministic given identical sample content/order and compatible software; however, metadata contains neither frozen indices nor a sample-content/order fingerprint. Its aggregate statistics cannot certify an unchanged row order. This addition therefore reports the saved split evidence but does not recreate it or claim to have checked live overlap when objects are absent. Before a fresh-kernel modelling run, human review must confirm the saved sample and replay of the existing split cell, or decide a split-persistence approach. No such design or replay is introduced here.

In [4]:
if "json" not in globals():
    import json
# Reuse the earlier metadata path; derive from the same sample location if absent/stale.
if "SPLIT_METADATA_PATH" not in globals() or not SPLIT_METADATA_PATH.is_file():
    SPLIT_METADATA_PATH = SAMPLE_PATH.resolve().parents[2] / "reports" / "modeling_split_metadata.json"
if not SPLIT_METADATA_PATH.is_file():
    raise FileNotFoundError(f"Existing split metadata not found: {SPLIT_METADATA_PATH}")
frozen_split_metadata = json.loads(SPLIT_METADATA_PATH.read_text())
assert frozen_split_metadata["modelling_sample_size"] == len(modeling_df)
assert frozen_split_metadata["features"] == FEATURES
assert frozen_split_metadata["target"] == TARGET
assert frozen_split_metadata["random_state"] == 42
assert frozen_split_metadata["test_size"] == 0.20
assert frozen_split_metadata["train_rows"] == 800_000
assert frozen_split_metadata["test_rows"] == 200_000
assert frozen_split_metadata["split_overlap"] == 0
assert retained_zero_distance_rows == frozen_split_metadata["sample_subgroup_counts"]["zero_distance"]
assert retained_high_fare_rows == frozen_split_metadata["sample_subgroup_counts"]["fare_above_100"]
print("Saved split: train=800,000; test=200,000; seed=42; test_size=0.20")
print("Saved overlap (historical result):", frozen_split_metadata["split_overlap"])
print("Index persistence:", frozen_split_metadata["index_note"])
print("Recorded software versions:", frozen_split_metadata["versions"])
print("Saved target summaries (historical results):")
display(pd.DataFrame(frozen_split_metadata["target_summary"]).set_index("split"))
existing_split_names = ["X_train", "X_test", "y_train", "y_test"]
split_objects_available = all(name in globals() for name in existing_split_names)
if split_objects_available:
    assert X_train.shape == (800_000, len(FEATURES))
    assert X_test.shape == (200_000, len(FEATURES))
    assert y_train.shape == (800_000,) and y_test.shape == (200_000,)
    assert X_train.index.equals(y_train.index)
    assert X_test.index.equals(y_test.index)
    current_split_overlap = len(X_train.index.intersection(X_test.index))
    assert current_split_overlap == 0
    reused_indices = X_train.index.append(X_test.index)
    assert reused_indices.is_unique and reused_indices.sort_values().equals(X.index.sort_values())
    assert X_train.equals(X.loc[X_train.index]) and X_test.equals(X.loc[X_test.index])
    assert y_train.equals(y.loc[y_train.index]) and y_test.equals(y.loc[y_test.index])
    del reused_indices
    print("Reused existing split objects; shapes, coverage, values, alignment and zero overlap verified.")
else:
    print("Existing split objects are not all available in this execution session.")
    print("No split was recreated. Saved counts/overlap are metadata evidence, not a live overlap check.")
    print("Human review: confirm sample identity/order and replay of the original split cell, or decide how to persist the frozen split.")
print("Integrity checks passed; no rows filtered and no feature values transformed.")
print("No model was trained. Stopped before model training for human review.")

Saved split: train=800,000; test=200,000; seed=42; test_size=0.20
Saved overlap (historical result): 0
Index persistence: Unique sample positions used only in memory; no indices or partition CSVs exported
Recorded software versions: {'numpy': '2.5.2', 'pandas': '3.0.5', 'sklearn': '1.9.0'}
Saved target summaries (historical results):
Existing split objects are not all available in this execution session.
No split was recreated. Saved counts/overlap are metadata evidence, not a live overlap check.
Human review: confirm sample identity/order and replay of the original split cell, or decide how to persist the frozen split.
Integrity checks passed; no rows filtered and no feature values transformed.
No model was trained. Stopped before model training for human review.


,rows,mean,median
split,,,
train,800000,11.268651,8.5
test,200000,11.264545,8.5


### Split Reproducibility Hardening

The Phase 3A split is deterministic only while the modeling sample content and row order remain unchanged. A file fingerprint and persistent split indices are therefore added before model training.

Replay the original unstratified `train_test_split` with `test_size=0.20`, `random_state=42` and `shuffle=True` on ordered sample positions. This makes the existing assignment explicit; it does not introduce a new sampling or splitting policy. Compare all available historical split statistics before writing any artifact. These comparisons corroborate the reconstruction; the new hashes establish an exact file/assignment reference from this point forward.

In [5]:
import hashlib
from sklearn.model_selection import train_test_split

HASH_CHUNK_BYTES = 1024 * 1024
FROZEN_TEST_SIZE = 0.20
FROZEN_RANDOM_STATE = 42
FROZEN_SHUFFLE = True
STATISTIC_ATOL = 1e-6
INDEX_DTYPE = np.dtype("<i8")  # Fixed-width int64, explicitly little-endian bytes.

def sha256_file_chunks(path, chunk_bytes=HASH_CHUNK_BYTES):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(chunk_bytes), b""):
            digest.update(block)
    return digest.hexdigest()

sample_stat_at_hash = SAMPLE_PATH.stat()
modeling_sample_sha256 = sha256_file_chunks(SAMPLE_PATH)
modeling_sample_size_bytes = sample_stat_at_hash.st_size
assert SAMPLE_PATH.stat().st_mtime_ns == sample_stat_at_hash.st_mtime_ns
print("Modeling sample path:", SAMPLE_PATH.resolve())
print("File size (bytes):", modeling_sample_size_bytes)
print("SHA256:", modeling_sample_sha256)
# Confirm the in-memory frame corresponds to the saved CSV in the same row order.
# CSV round-trip differences in floating-point representation are tolerated, not row changes.
verified_rows = 0
with pd.read_csv(SAMPLE_PATH, chunksize=100_000) as reader:
    for block in reader:
        expected_block = modeling_df.iloc[verified_rows:verified_rows + len(block)]
        assert list(block.columns) == list(modeling_df.columns)
        assert len(block) == len(expected_block)
        np.testing.assert_allclose(block.to_numpy(), expected_block.to_numpy(), rtol=0, atol=1e-12)
        verified_rows += len(block)
assert verified_rows == len(modeling_df)
assert X.equals(modeling_df[FEATURES]) and y.equals(modeling_df[TARGET])
print("Saved-file row order and in-memory modeling values verified.")

Modeling sample path: /Users/xiaochuan/nyc-taxi-fare-prediction/data/processed/modeling_sample_1m.csv
File size (bytes): 75078404
SHA256: c0a1f2eba4a90d1581faeaa707d7ae869423a8c5f76c337b6024a3c9fc933aef
Saved-file row order and in-memory modeling values verified.


In [6]:
all_indices = np.arange(len(modeling_df), dtype=INDEX_DTYPE)
train_idx, test_idx = train_test_split(
    all_indices,
    test_size=FROZEN_TEST_SIZE,
    random_state=FROZEN_RANDOM_STATE,
    shuffle=FROZEN_SHUFFLE,
)
train_idx = np.asarray(train_idx, dtype=INDEX_DTYPE)
test_idx = np.asarray(test_idx, dtype=INDEX_DTYPE)
assert len(train_idx) == 800_000
assert len(test_idx) == 200_000
split_overlap = len(np.intersect1d(train_idx, test_idx))
assert split_overlap == 0
assert np.unique(train_idx).size == len(train_idx)
assert np.unique(test_idx).size == len(test_idx)
assert np.union1d(train_idx, test_idx).size == len(modeling_df) == 1_000_000
for indices in (train_idx, test_idx):
    assert indices.min() >= 0 and indices.max() < len(modeling_df)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
assert X_train.shape == (800_000, 9)
assert X_test.shape == (200_000, 9)
assert y_train.shape == (800_000,)
assert y_test.shape == (200_000,)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
print("Reconstructed shapes:", X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print("Split overlap:", split_overlap)

Reconstructed shapes: (800000, 9) (200000, 9) (800000,) (200000,)
Split overlap: 0


In [7]:
# Read the existing JSON structure rather than substituting rounded historical values.
metadata_before_text = SPLIT_METADATA_PATH.read_text()
metadata_before_hardening = json.loads(metadata_before_text)
historical_targets = {row["split"]: row for row in metadata_before_hardening["target_summary"]}
comparison_records = []

def record_match(check, actual, expected, floating=False):
    matches = bool(np.isclose(actual, expected, rtol=0, atol=STATISTIC_ATOL)) if floating else bool(actual == expected)
    comparison_records.append({"check": check, "reconstructed": actual, "historical": expected, "match": matches})

record_match("train rows", len(train_idx), metadata_before_hardening["train_rows"])
record_match("test rows", len(test_idx), metadata_before_hardening["test_rows"])
record_match("sample rows", len(modeling_df), metadata_before_hardening["modelling_sample_size"])
record_match("random_state", FROZEN_RANDOM_STATE, metadata_before_hardening["random_state"])
record_match("test_size", FROZEN_TEST_SIZE, metadata_before_hardening["test_size"])
record_match("overlap", split_overlap, metadata_before_hardening["split_overlap"])
for label, target in [("train", y_train), ("test", y_test)]:
    record_match(f"{label} target rows", len(target), historical_targets[label]["rows"])
    record_match(f"{label} target mean", float(target.mean()), historical_targets[label]["mean"], floating=True)
    record_match(f"{label} target median", float(target.median()), historical_targets[label]["median"], floating=True)
metadata_comparison = pd.DataFrame(comparison_records)
display(metadata_comparison)
phase3a_metadata_match = bool(metadata_comparison["match"].all())
if not phase3a_metadata_match:
    failures = metadata_comparison.loc[~metadata_comparison["match"]]
    raise ValueError("Reconstruction mismatch; no persistence permitted:\n" + failures.to_string(index=False))
assert metadata_before_hardening["features"] == FEATURES
assert metadata_before_hardening["target"] == TARGET
print("Reconstructed split matches Phase 3A metadata.")
# Check all modelling arrays before persistence; no preprocessing is performed.
for name, values in [("X", X), ("y", y), ("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test)]:
    assert not values.isna().to_numpy().any(), name
    assert np.isfinite(values.to_numpy()).all(), name
assert TARGET not in X_train.columns and TARGET not in X_test.columns

Reconstructed split matches Phase 3A metadata.


,check,reconstructed,historical,match
0,train rows,800000.000000,800000.000000,True
1,test rows,200000.000000,200000.000000,True
2,sample rows,1000000.000000,1000000.000000,True
3,random_state,42.000000,42.000000,True
4,test_size,0.200000,0.200000,True
5,overlap,0.000000,0.000000,True
6,train target rows,800000.000000,800000.000000,True
7,train target mean,11.268651,11.268651,True
8,train target median,8.500000,8.500000,True
9,test target rows,200000.000000,200000.000000,True


#### Persistent Artifact and Verification Contract

Store `train_idx` and `test_idx` as ordered int64 sample positions in the ignored processed-data directory. Their byte hashes use explicit little-endian int64 representation and include index order. Reload the NPZ and require exact array equality before updating metadata.

For future reuse, first match the CSV SHA256 and both index hashes against metadata, then select X/y with `.iloc`. A mismatch requires review rather than silently generating another split. Original metadata fields, including the historical note that indices had not been exported at preparation time, are retained; the new `reproducibility` section records their persistence now.

In [8]:
assert phase3a_metadata_match
sample_project_directory = SAMPLE_PATH.resolve().parents[2]
assert "data/processed/" in [line.strip() for line in (sample_project_directory / ".gitignore").read_text().splitlines()]
SPLIT_INDICES_PATH = SAMPLE_PATH.parent / "modeling_split_indices.npz"
train_indices_sha256 = hashlib.sha256(np.asarray(train_idx, dtype=INDEX_DTYPE).tobytes()).hexdigest()
test_indices_sha256 = hashlib.sha256(np.asarray(test_idx, dtype=INDEX_DTYPE).tobytes()).hexdigest()
# Recheck source identity and concurrent metadata edits immediately before writing.
assert sha256_file_chunks(SAMPLE_PATH) == modeling_sample_sha256
assert SPLIT_METADATA_PATH.read_text() == metadata_before_text
# Reruns may reuse an identical artifact, but never silently overwrite different assignments.
if SPLIT_INDICES_PATH.exists():
    with np.load(SPLIT_INDICES_PATH, allow_pickle=False) as existing:
        assert set(existing.files) == {"train_idx", "test_idx"}
        assert np.array_equal(existing["train_idx"], train_idx)
        assert np.array_equal(existing["test_idx"], test_idx)
else:
    np.savez_compressed(SPLIT_INDICES_PATH, train_idx=train_idx, test_idx=test_idx)
with np.load(SPLIT_INDICES_PATH, allow_pickle=False) as loaded:
    assert loaded["train_idx"].dtype == INDEX_DTYPE
    assert loaded["test_idx"].dtype == INDEX_DTYPE
    assert np.array_equal(loaded["train_idx"], train_idx)
    assert np.array_equal(loaded["test_idx"], test_idx)
    assert hashlib.sha256(loaded["train_idx"].tobytes()).hexdigest() == train_indices_sha256
    assert hashlib.sha256(loaded["test_idx"].tobytes()).hexdigest() == test_indices_sha256
persistent_split_artifact_verified = True
print("Local split artifact:", SPLIT_INDICES_PATH.resolve())
print("Persistent split artifact verified:", persistent_split_artifact_verified)
print("Train index SHA256:", train_indices_sha256)
print("Test index SHA256:", test_indices_sha256)

Local split artifact: /Users/xiaochuan/nyc-taxi-fare-prediction/data/processed/modeling_split_indices.npz
Persistent split artifact verified: True
Train index SHA256: 3a708f8187d5137d9fa78fee5abaa8dbd7db6dca3cd48fe6c103b43dec9e6df3
Test index SHA256: 3514db48cdb7a5a86e8cd5bb038eccb4b8c55867a095288c63b89dce367b756f


In [9]:
assert phase3a_metadata_match and persistent_split_artifact_verified
reproducibility_record = {
    "modeling_sample_sha256": modeling_sample_sha256,
    "modeling_sample_size_bytes": modeling_sample_size_bytes,
    "split_indices_artifact": "data/processed/modeling_split_indices.npz",
    "train_indices_sha256": train_indices_sha256,
    "test_indices_sha256": test_indices_sha256,
    "index_dtype": "int64",
    "index_byte_order": "little",
    "index_semantics": "Ordered zero-based row positions in the fingerprinted modeling sample CSV",
    "split_algorithm": "sklearn.model_selection.train_test_split",
    "shuffle": FROZEN_SHUFFLE,
    "test_size": FROZEN_TEST_SIZE,
    "random_state": FROZEN_RANDOM_STATE,
    "phase3a_metadata_match": phase3a_metadata_match,
    "mean_comparison_atol": STATISTIC_ATOL,
}
# On rerun, keep any extra existing reproducibility fields; stop if core identity changed.
if "reproducibility" in metadata_before_hardening:
    previous = metadata_before_hardening["reproducibility"]
    for key in ["modeling_sample_sha256", "train_indices_sha256", "test_indices_sha256"]:
        assert previous.get(key) == reproducibility_record[key]
    reproducibility_record = {**previous, **reproducibility_record}
updated_split_metadata = {**metadata_before_hardening, "reproducibility": reproducibility_record}
assert SPLIT_METADATA_PATH.read_text() == metadata_before_text
SPLIT_METADATA_PATH.write_text(json.dumps(updated_split_metadata, indent=2) + "\n")
reloaded_split_metadata = json.loads(SPLIT_METADATA_PATH.read_text())
for key, value in metadata_before_hardening.items():
    if key != "reproducibility":
        assert reloaded_split_metadata[key] == value
assert reloaded_split_metadata["reproducibility"] == reproducibility_record
print("Updated split metadata; all original fields preserved.")

Updated split metadata; all original fields preserved.


In [10]:
assert TARGET not in X_train.columns and TARGET not in X_test.columns
for values in (X, y, X_train, X_test, y_train, y_test):
    assert not values.isna().to_numpy().any()
    assert np.isfinite(values.to_numpy()).all()
print("Modeling sample SHA256:", modeling_sample_sha256)
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Split overlap:", split_overlap)
print("Train index SHA256:", train_indices_sha256)
print("Test index SHA256:", test_indices_sha256)
print("Phase 3A metadata match:", phase3a_metadata_match)
print("Persistent split artifact verified:", persistent_split_artifact_verified)
print("No model was trained. The Phase 3A split is now reproducibly frozen and ready for Phase 3B model training.")

Modeling sample SHA256: c0a1f2eba4a90d1581faeaa707d7ae869423a8c5f76c337b6024a3c9fc933aef
Train rows: 800000
Test rows: 200000
Split overlap: 0
Train index SHA256: 3a708f8187d5137d9fa78fee5abaa8dbd7db6dca3cd48fe6c103b43dec9e6df3
Test index SHA256: 3514db48cdb7a5a86e8cd5bb038eccb4b8c55867a095288c63b89dce367b756f
Phase 3A metadata match: True
Persistent split artifact verified: True
No model was trained. The Phase 3A split is now reproducibly frozen and ready for Phase 3B model training.
